# Portfolio Project: SpaceX Falcon 9 First-Stage Landing Prediction
## 03 — Data Wrangling and Target Construction

**Portfolio Project | Business Analytics & Data Analytics**

This notebook transforms the launch-level dataset created in Notebook 01 into the **modeling-ready analytical dataset** used throughout the remainder of the project. The main task is to convert the detailed landing outcome field into a transparent **binary target** indicating whether the Falcon 9 first stage landed successfully.

### Research role of this stage
Data wrangling is not simply a formatting exercise. The definition of the response variable determines what the later exploratory and machine-learning analyses are actually predicting. For that reason, the target-construction rule is made explicit, validated against the observed outcome categories, and checked for class balance before export.

### Workflow
1. Load the historical Falcon 9 launch dataset from the local project file.
2. Validate the expected schema and observation count.
3. Audit missingness and data types.
4. Examine launch-site, orbit, and landing-outcome distributions.
5. Define success and failure outcomes using an explicit semantic rule.
6. Create the binary `Class` target.
7. Validate target consistency and class balance.
8. Export the modeling-ready dataset as `dataset_part_2_portfolio.csv`.

> **Note.** The resulting `Class` definition: `1` represents a successful first-stage landing and `0` represents an unsuccessful or unavailable landing outcome.

## 1. Environment and local data source

The dataset is loaded from the **local project directory** fetched from notebook 01, which makes the analytical workflow self-contained.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

INPUT_PATH = Path("dataset_part_1.csv")
OUTPUT_PATH = Path("dataset_part_2_portfolio.csv")

EXPECTED_COLUMNS = [
    "FlightNumber", "Date", "BoosterVersion", "PayloadMass", "Orbit",
    "LaunchSite", "Outcome", "Flights", "GridFins", "Reused", "Legs",
    "LandingPad", "Block", "ReusedCount", "Serial", "Longitude", "Latitude"
]

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH} was not found. Place it in the same directory as this notebook."
    )

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns from {INPUT_PATH}")

Loaded 90 rows × 17 columns from dataset_part_1.csv


## 2. Structural validation

Before deriving the target variable, the input dataset is checked against the expected 17-column schema from Notebook 01. This catches accidental file substitutions or upstream structural changes before they propagate into the analysis.

In [2]:
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))

validation = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "missing_expected_columns": len(missing_columns),
    "unexpected_columns": len(unexpected_columns),
    "duplicate_rows": int(df.duplicated().sum()),
    "unique_flight_numbers": df["FlightNumber"].nunique(),
}, name="value")

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

validation.to_frame()

,value
rows,90
columns,17
missing_expected_columns,0
unexpected_columns,0
duplicate_rows,0
unique_flight_numbers,90


In [3]:
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


### Date and type normalization

`Date` is converted to a pandas datetime type for more reliable downstream filtering and validation. The remaining columns are inspected rather than aggressively coerced because booleans, identifiers, and categorical fields carry different analytical meanings.

In [4]:
df["Date"] = pd.to_datetime(df["Date"], errors="raise")

dtype_summary = df.dtypes.astype(str).rename("dtype").to_frame()
dtype_summary

,dtype
FlightNumber,int64
Date,datetime64[ns]
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


## 3. Missingness audit

Notebook 01 already imputed missing payload mass. The remaining missingness is concentrated in `LandingPad`, which is not automatically treated as an error because some missions did not use a landing pad. This distinction is important: not every `NaN` should be imputed without considering operational meaning.

In [5]:
missingness = df.isna().sum().rename("missing_count").to_frame()
missingness["missing_pct"] = (
    100 * missingness["missing_count"] / len(df)
).round(2)

missingness

,missing_count,missing_pct
FlightNumber,0,0.00
Date,0,0.00
BoosterVersion,0,0.00
PayloadMass,0,0.00
Orbit,0,0.00
LaunchSite,0,0.00
Outcome,0,0.00
Flights,0,0.00
GridFins,0,0.00
Reused,0,0.00


## 4. Descriptive checks before target construction

We examine launch-site, orbit, and landing-outcome frequencies before defining the target. Those checks are important because they provide useful context for the sample and help verify that the categorical fields were imported correctly.

### 4.1 Launches by site

In [6]:
launch_site_counts = (
    df["LaunchSite"]
    .value_counts()
    .rename_axis("LaunchSite")
    .to_frame("Launches")
)

launch_site_counts

,Launches
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13


The launch-site distribution shows how the 90 historical Falcon 9 missions are allocated across the three sites used in this project. This imbalance is relevant later because launch site may be associated with mission profile and landing conditions.

### 4.2 Launches by orbit

The full orbit distribution is reported first. The original course exercise also displayed a version excluding `GTO` because geostationary transfer orbit is a transfer trajectory rather than a final geostationary orbit. For transparency, both views are retained rather than silently removing those missions.

In [7]:
orbit_counts = (
    df["Orbit"]
    .value_counts()
    .rename_axis("Orbit")
    .to_frame("Launches")
)

orbit_counts

,Launches
Orbit,
GTO,27
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
HEO,1
ES-L1,1


In [8]:
orbit_counts_excluding_gto = orbit_counts.drop(index="GTO", errors="ignore")
orbit_counts_excluding_gto

,Launches
Orbit,
ISS,21
VLEO,14
PO,9
LEO,7
SSO,5
MEO,3
HEO,1
ES-L1,1
SO,1


Each launch aims to an dedicated orbit, and here are some common orbit types:


* <b>LEO</b>: Low Earth orbit (LEO)is an Earth-centred orbit with an altitude of 2,000 km (1,200 mi) or less (approximately one-third of the radius of Earth),[1] or with at least 11.25 periods per day (an orbital period of 128 minutes or less) and an eccentricity less than 0.25.[2] Most of the manmade objects in outer space are in LEO <a href='https://en.wikipedia.org/wiki/Low_Earth_orbit'>[1]</a>.

* <b>VLEO</b>: Very Low Earth Orbits (VLEO) can be defined as the orbits with a mean altitude below 450 km. Operating in these orbits can provide a number of benefits to Earth observation spacecraft as the spacecraft operates closer to the observation<a href='https://www.researchgate.net/publication/271499606_Very_Low_Earth_Orbit_mission_concepts_for_Earth_Observation_Benefits_and_challenges'>[2]</a>.


* <b>GTO</b>(Geostationary Transfer Orbit): A geostationary transfer orbit is an elliptical Earth orbit used to transfer satellites from low Earth orbit (LEO) to geostationary orbit (GEO). In a GTO, the perigee (closest point to Earth) is much lower than GEO altitude, while the apogee (farthest point) reaches approximately 22,236 miles (35,786 kilometers) above Earth’s equator — the altitude of a geostationary orbit. Satellites in GTO use onboard propulsion to circularize their orbit at GEO altitude, where they can provide services such as weather monitoring, communications, and surveillance. <a  href="https://www.space.com/29222-geosynchronous-orbit.html" >[3] </a>.


* <b>SSO (or SO)</b>: It is a Sun-synchronous orbit  also called a heliosynchronous orbit is a nearly polar orbit around a planet, in which the satellite passes over any given point of the planet's surface at the same local mean solar time <a href="https://en.wikipedia.org/wiki/Sun-synchronous_orbit">[4] <a>.
    
    
* <b>ES-L1 </b>:At the Lagrange points the gravitational forces of the two large bodies cancel out in such a way that a small object placed in orbit there is in equilibrium relative to the center of mass of the large bodies. L1 is one such point between the sun and the earth <a href="https://en.wikipedia.org/wiki/Lagrange_point#L1_point">[5]</a> .
    
    
* <b>HEO</b> A highly elliptical orbit, is an elliptic orbit with high eccentricity, usually referring to one around Earth <a href="https://en.wikipedia.org/wiki/Highly_elliptical_orbit">[6]</a>.


* <b> ISS </b> A modular space station (habitable artificial satellite) in low Earth orbit. It is a multinational collaborative project between five participating space agencies: NASA (United States), Roscosmos (Russia), JAXA (Japan), ESA (Europe), and CSA (Canada)<a href="https://en.wikipedia.org/wiki/International_Space_Station"> [7] </a>


* <b> MEO </b> Geocentric orbits ranging in altitude from 2,000 km (1,200 mi) to just below geosynchronous orbit at 35,786 kilometers (22,236 mi). Also known as an intermediate circular orbit. These are "most commonly at 20,200 kilometers (12,600 mi), or 20,650 kilometers (12,830 mi), with an orbital period of 12 hours <a href="https://en.wikipedia.org/wiki/List_of_orbits"> [8] </a>


* <b> HEO </b> Geocentric orbits above the altitude of geosynchronous orbit (35,786 km or 22,236 mi) <a href="https://en.wikipedia.org/wiki/List_of_orbits"> [9] </a>


* <b> GEO </b> It is a circular geosynchronous orbit 35,786 kilometres (22,236 miles) above Earth's equator and following the direction of Earth's rotation <a href="https://en.wikipedia.org/wiki/Geostationary_orbit"> [10] </a>


* <b> PO </b> It is one type of satellites in which a satellite passes above or nearly above both poles of the body being orbited (usually a planet such as the Earth <a href="https://en.wikipedia.org/wiki/Polar_orbit"> [11] </a>

some are shown in the following plot:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/Orbits.png)


### 4.3 Detailed landing outcomes

The `Outcome` field combines landing success state with landing method/location:

- `True ASDS` — successful landing on an Autonomous Spaceport Drone Ship;
- `True RTLS` — successful Return-to-Launch-Site landing;
- `True Ocean` — successful controlled ocean landing;
- `False ...` — attempted but unsuccessful landing;
- `None ...` — no successful landing outcome recorded.

These detailed categories are useful descriptively, but the predictive task requires a binary target.

In [9]:
landing_outcomes = (
    df["Outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .to_frame("Count")
)

landing_outcomes

,Count
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


## 5. Construct the binary landing-success target

We define the landing-sucess target **semantically**:

- an outcome beginning with `True` → successful landing (`Class = 1`);
- an outcome beginning with `False` or `None` → unsuccessful/no successful landing (`Class = 0`).

This makes the target definition explicit, auditable, and stable with respect to category frequency ordering.

In [10]:
observed_outcomes = sorted(df["Outcome"].dropna().unique())
observed_outcomes

['False ASDS',
 'False Ocean',
 'False RTLS',
 'None ASDS',
 'None None',
 'True ASDS',
 'True Ocean',
 'True RTLS']

In [11]:
def landing_success_label(outcome):
    """Map detailed landing outcome text to the project's binary target."""
    if pd.isna(outcome):
        raise ValueError("Outcome is missing; target cannot be assigned unambiguously.")

    outcome = str(outcome).strip()

    if outcome.startswith("True "):
        return 1
    if outcome.startswith(("False ", "None ")):
        return 0

    raise ValueError(f"Unexpected landing outcome category: {outcome!r}")


df["Class"] = df["Outcome"].map(landing_success_label).astype("int8")

df[["Outcome", "Class"]].head(10)

,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


## 6. Validate the target definition

A target should not be accepted simply because the code runs. The checks below verify that:

1. every observed outcome category maps to exactly one binary class;
2. all `True ...` outcomes map to `1`;
3. all `False ...` and `None ...` outcomes map to `0`;
4. the resulting class distribution matches the historical project result.

In [12]:
outcome_class_map = (
    df.groupby("Outcome", dropna=False)["Class"]
      .agg(["count", "min", "max"])
      .sort_values("count", ascending=False)
)

outcome_class_map

,count,min,max
Outcome,,,
True ASDS,41,1,1
None None,19,0,0
True RTLS,14,1,1
False ASDS,6,0,0
True Ocean,5,1,1
False Ocean,2,0,0
None ASDS,2,0,0
False RTLS,1,0,0


In [13]:
assert (outcome_class_map["min"] == outcome_class_map["max"]).all()
assert df.loc[df["Outcome"].str.startswith("True "), "Class"].eq(1).all()
assert df.loc[df["Outcome"].str.startswith(("False ", "None ")), "Class"].eq(0).all()

print("Target validation passed: every observed landing outcome is mapped consistently.")

Target validation passed: every observed landing outcome is mapped consistently.


## 7. Class balance and historical success rate

Class balance matters because raw accuracy can be misleading when one class is substantially more common than the other. This becomes especially important in Notebook 08, where multiple classification models are compared.

In [14]:
class_summary = (
    df["Class"]
    .value_counts()
    .sort_index()
    .rename_axis("Class")
    .to_frame("Count")
)

class_summary["Share"] = (class_summary["Count"] / len(df)).round(4)
class_summary.index = ["Failure (0)", "Success (1)"]

class_summary

,Count,Share
Failure (0),30,0.3333
Success (1),60,0.6667


In [15]:
success_rate = df["Class"].mean()

print(f"Historical landing success rate: {success_rate:.2%}")
print(f"Successful landings: {df['Class'].sum()} of {len(df)}")
print(f"Unsuccessful/no-success outcomes: {(df['Class'] == 0).sum()} of {len(df)}")

Historical landing success rate: 66.67%
Successful landings: 60 of 90
Unsuccessful/no-success outcomes: 30 of 90


### Interpretation

The historical sample contains **60 successful landings and 30 unsuccessful/no-success outcomes**, corresponding to an overall success rate of **66.67%**.

This 2:1 class distribution is not extreme, but it is sufficiently **imbalanced** that later predictive evaluation should not rely on accuracy alone. Precision, recall, F1, balanced accuracy, ROC-AUC, and confusion-matrix behavior will therefore be considered in the upgraded modeling notebook.

## 8. Final dataset validation

The final table should contain the original 17 launch-level variables plus the newly constructed `Class` target. The checks below verify row count, column count, target completeness, and key uniqueness before export.

In [16]:
final_validation = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "unique_flight_numbers": df["FlightNumber"].nunique(),
    "missing_target_values": int(df["Class"].isna().sum()),
    "target_classes": df["Class"].nunique(),
    "successes": int(df["Class"].sum()),
    "failures": int((df["Class"] == 0).sum()),
}, name="value")

final_validation.to_frame()

,value
rows,90
columns,18
unique_flight_numbers,90
missing_target_values,0
target_classes,2
successes,60
failures,30


In [17]:
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## 9. Export the modeling-ready dataset

The portfolio version writes to `dataset_part_2_portfolio.csv`. The schema and target definition are compatible with the dataset used by the subsequent notebooks.

In [18]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df):,} rows × {df.shape[1]} columns to: {OUTPUT_PATH.resolve()}")

Saved 90 rows × 18 columns to: /mnt/data/dataset_part_2_portfolio.csv


## 10. Key takeaways

This wrangling stage converts the descriptive landing outcome into the **binary response variable** used throughout the rest of the portfolio.

### Analytical handoff

The resulting dataset contains **90 Falcon 9 missions and 18 variables**, including the binary `Class` target. It becomes the core input to the SQL EDA, visualization, geospatial, dashboard, and classification stages that follow.

## Project attribution

This portfolio notebook builds on the data-wrangling stage of the **IBM Data Science Professional Certificate SpaceX capstone**. The historical analytical objective and class definition are retained, while the implementation has been refactored to improve robustness, transparency, validation, and research-oriented interpretation.